In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from pathlib import Path

RESULT_DIRS = [Path('../../results')]

# Colorblind-safe Wong palette
COLORS = {'Baseline': '#0072B2', 'SGX': '#D55E00', 'Confidential5G': '#009E73'}
LABELS = {'Baseline': 'Baseline', 'SGX': 'C5G-SGX', 'Confidential5G': 'C5G-TDX'}
HATCHES = {'Baseline': '', 'SGX': 'xx', 'Confidential5G': '//'}
FIG_H  = 2.4
COL_W  = 3.33
TEXT_W = 7.16
FONT   = {'tick': 10, 'label': 10, 'title': 10, 'legend': 8}

plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Times', 'Nimbus Roman', 'DejaVu Serif'],
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.linewidth': 0.8,
    'grid.linewidth': 0.5,
    'hatch.linewidth': 0.6,
})

In [ ]:
RD     = RESULT_DIRS[0]
IMAGES = Path('../images')
IMAGES.mkdir(exist_ok=True)

# File selection - edit these to change which run is analysed
NF_FILES = {
    'Baseline':       RD / 'cp_scalability_baseline_concurrent_100_overhead_nf.csv',
    'SGX':            RD / 'cp_scalability_sgx_concurrent_100_overhead_nf.csv',
    'Confidential5G': RD / 'cp_scalability_hybrid_concurrent_100_2_overhead_nf.csv',
}
SYS_FILES = {
    'Baseline':       RD / 'cp_scalability_baseline_concurrent_100_overhead_sys.csv',
    'SGX':            RD / 'cp_scalability_sgx_concurrent_100_overhead_sys.csv',
    'Confidential5G': RD / 'cp_scalability_hybrid_concurrent_100_2_overhead_sys.csv',
}
UPF_FILES = {
    'Baseline':       RD / 'up_scalability_baseline_ping_200_overhead_nf.csv',
    'SGX':            RD / 'up_scalability_sgx_ping_200_3_overhead_nf.csv',
    'Confidential5G': RD / 'up_scalability_hybrid_ping_200_3_overhead_nf.csv',
}

NFS  = ['amf', 'ausf', 'pcf', 'scp', 'smf', 'udm', 'upf']
DEPS = ['Baseline', 'Confidential5G', 'SGX']

nf_dfs  = {d: pd.read_csv(p) for d, p in NF_FILES.items()}
sys_dfs = {d: pd.read_csv(p) for d, p in SYS_FILES.items()}
upf_dfs = {d: pd.read_csv(p) for d, p in UPF_FILES.items()}

def phase(df, name):
    if 'phase' in df.columns and name in df['phase'].values:
        return df[df['phase'] == name]
    return df

def p95(df, col):
    if col not in df.columns: return np.nan
    return pd.to_numeric(df[col], errors='coerce').quantile(.95)

def med(df, col):
    if col not in df.columns: return np.nan
    return pd.to_numeric(df[col], errors='coerce').median()

def mean_nz(df, col):
    if col not in df.columns: return np.nan
    s = pd.to_numeric(df[col], errors='coerce')
    nz = s[s > 0]
    return round(nz.mean(), 1) if len(nz) else np.nan

print('Loaded files:')
for d in DEPS:
    df = nf_dfs[d]
    phases = df['phase'].value_counts().to_dict() if 'phase' in df.columns else 'no phase column'
    print(f'  {d}: {NF_FILES[d].name}  phases={phases}')

In [ ]:
# Figure 1: Per-NF CPU Utilization - P95

cpu_deps = ['Baseline', 'Confidential5G', 'SGX']

fig, ax = plt.subplots(figsize=(COL_W, FIG_H))
x = np.arange(len(NFS))
w = 0.22

for i, dep in enumerate(cpu_deps):
    vals = []
    for nf in NFS:
        df = upf_dfs[dep] if nf == 'upf' else nf_dfs[dep]
        vals.append(p95(df, f'{nf}_cpu_pct'))
    off = (i - 1) * w
    ax.bar(x + off, vals, width=w, color=COLORS[dep], label=LABELS[dep],
           hatch=HATCHES[dep], edgecolor='white', linewidth=0.6, zorder=3)

ax.set_xticks(x)
ax.set_xticklabels([n.upper() for n in NFS], fontsize=FONT['tick'])
ax.set_ylabel('CPU Utilization (%)', fontsize=FONT['label'])
ax.tick_params(axis='both', labelsize=FONT['tick'])
ax.legend(fontsize=FONT['legend'], frameon=False, loc='upper left')
ax.grid(True, axis='y', linestyle='-', alpha=0.15, zorder=0)
ax.set_ylim(bottom=0)
ax.set_xlim(-0.5, len(NFS) - 0.5)
fig.tight_layout()
fig.savefig(IMAGES / 'resource_cpu.png', bbox_inches='tight', dpi=300)
fig.savefig(IMAGES / 'resource_cpu.pdf', bbox_inches='tight')
plt.show()

In [ ]:
from IPython.display import display

def _fmt(v, fallback='[ - ]'):
    return fallback if (isinstance(v, float) and np.isnan(v)) else v

def _ratio(base, new):
    if np.isnan(base) or np.isnan(new) or base == 0: return np.nan
    return round(float(new) / float(base), 2)

# Table A: SGX Enclave Overhead Per NF (registration phase)
b_reg = phase(nf_dfs['Baseline'], 'registration')
s_reg = phase(nf_dfs['SGX'],      'registration')

rows_a = []
for nf in NFS:
    rows_a.append({
        'NF':                    nf.upper(),
        'EENTER/s':              _fmt(mean_nz(s_reg, f'{nf}_eenter_rate')),
        'EEXIT/s':               _fmt(mean_nz(s_reg, f'{nf}_eexit_rate')),
        'AEX/s':                 _fmt(mean_nz(s_reg, f'{nf}_aex_rate')),
        'IPC (Base)':            _fmt(med(b_reg, f'{nf}_ipc')),
        'IPC (SGX)':             _fmt(med(s_reg, f'{nf}_ipc')),
        'IPC ratio':             _fmt(_ratio(med(b_reg, f'{nf}_ipc'), med(s_reg, f'{nf}_ipc'))),
        'Cache miss % (Base)':   _fmt(round(med(b_reg, f'{nf}_cache_miss_pct'), 1)),
        'Cache miss % (SGX)':    _fmt(round(med(s_reg, f'{nf}_cache_miss_pct'), 1)),
    })

print('Table A: SGX Enclave Overhead - Registration Phase @ 100 UEs')
print('(EENTER/EEXIT/AEX = mean rate during active windows; IPC/cache = median)')
display(pd.DataFrame(rows_a).set_index('NF'))

# Table B: TDX VM Exit Breakdown (registration phase)
EXIT_TYPES = ['halt', 'hypercall', 'io', 'irq']
TDX_VMS = {
    'SGX':            [('SGX – tdx_amf',    'tdx_amf')],
    'Confidential5G': [('Conf5G – tdx_amf', 'tdx_amf'), ('Conf5G – tdx_upf', 'tdx_upf')],
}

rows_b = []
for dep, vms in TDX_VMS.items():
    df_sys = phase(sys_dfs[dep], 'registration')
    for label, prefix in vms:
        total_col = f'{prefix}_vmexit_rate'
        if total_col not in df_sys.columns: continue
        total = med(df_sys, total_col)
        row = {'TDX VM': label, 'Total exits/s': round(total, 1)}
        known = 0.0
        for t in EXIT_TYPES:
            c = f'{prefix}_vmexit_{t}_rate'
            v = med(df_sys, c) if c in df_sys.columns else 0.0
            row[f'{t.capitalize()}/s'] = round(v, 1); known += float(v)
        row['Other/s'] = round(max(float(total) - known, 0.0), 1)
        rows_b.append(row)

print('\nTable B: TDX VM Exit Breakdown - Registration Phase @ 100 UEs')
if rows_b:
    display(pd.DataFrame(rows_b).set_index('TDX VM'))
else:
    print('No VM-exit columns found.')